In [ ]:
import torch
from torch import nn

RTOL = 1e-5
ATOL = 1e-8


class iResDemo(nn.Module):
    def __init__(self, input_size: int, debug: bool = True) -> None:
        super().__init__()

        self.input_size = input_size
        self.debug = debug

        # initialize A with spectral norm < 1

        A = torch.randn(input_size, input_size)
        A = A / (torch.linalg.matrix_norm(A, ord=2) + 0.05)
        self.A = nn.Parameter(A)

        # self.register_buffer("x", torch.empty(()))
        # self.register_buffer("y", torch.empty(()))

    def f(self, x: torch.Tensor) -> torch.Tensor:
        return torch.einsum("ij, ...i -> ...j", self.A, x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.x = x
        y = x + self.f(x)
        self.y = y
        return y

    def inverse(self, y: torch.Tensor, n_iter: int = 1000) -> torch.Tensor:
        with torch.no_grad():
            # x = self.x.detach().clone().requires_grad_(True)
            x = torch.zeros_like(y)
            assert x.grad is None
            assert y.grad is None
            # assert not x.requires_grad

        assert x.shape == y.shape, f"Expected shape {y.shape}, got {x.shape}"
        for k in range(n_iter):
            x_next = y - self.f(x)

            # check for convergence
            if torch.linalg.norm(x_next - x) < RTOL * torch.linalg.norm(x) + ATOL:
                if self.debug:
                    print("Inverse converged in", k, "iterations")
                break
            x = x_next
        else:
            if self.debug:
                print(
                    "Warning: Inverse did not converge within the maximum number of iterations"
                )

        # self.x = x
        return x

    def inverse_with_manual_grad(
        self, y: torch.Tensor, n_iter: int = 1000
    ) -> torch.Tensor:
        with torch.no_grad():
            x = self.inverse(y, n_iter)
        assert x.grad is None

        # perform 1 more iter with grad enabled
        x = y - self.f(x)

        # set up Jacobian vector product (without additional forward calls)
        x0 = x.clone().detach().requires_grad_()
        f0 = y - self.f(x0)

        def backward_hook(grad):
            if self.debug:
                print("Custom backward hook called")
                print(f"{grad=}, {type(grad)}")

            if grad is None:
                return None

            if len(x.shape) == 2:
                G = torch.stack(
                    [
                        torch.autograd.functional.jacobian(
                            lambda x: y[k] - self.f(x), x[k], create_graph=True
                        )
                        for k in range(len(x))
                    ]
                )
            else:
                G = torch.autograd.functional.jacobian(
                    lambda x: y - self.f(x), x, create_graph=True
                )

            if self.debug:
                print(f"{G.shape=}")
            # setup linear system to solve (I - J_f)ᵀ g = grad
            R = torch.eye(self.input_size) - G.swapaxes(-2, -1)
            g = torch.linalg.solve(R, grad)
            return g

            g, self.backward_res = self.solver(
                lambda outer_grad: (
                    autograd.grad(f0, x0, outer_grad, retain_graph=True)[0] + grad
                ),
                grad,
            )
            return g

        x.register_hook(backward_hook)
        return x

In [ ]:
n = 4

loss = nn.MSELoss()
model = iResDemo(input_size=n)
x_orig = torch.randn(3, n)
y = model(x_orig)
with torch.no_grad():
    y = y.clone()
y

In [ ]:
model.zero_grad()
assert model.A.grad is None
x = model.inverse(y)
x.norm().backward()
model.A.grad

In [ ]:
# re-enage gradients by computing x+f(x)
model.zero_grad()
assert model.A.grad is None
x = model.inverse_with_manual_grad(y)
x.norm().backward()
model.A.grad

In [ ]:
y = torch.randn(4).requires_grad_()
# model.inverse_with_manual_grad(y)
torch.autograd.gradcheck(model.inverse_with_manual_grad, (y,), eps=1e-4, atol=1e-3)

In [ ]:
def test_grad():
    y = torch.randn(3, n)

    # compute original grad
    model.zero_grad()
    assert model.A.grad is None
    x = model.inverse(y)
    x.norm().backward()
    orig_grad = model.A.grad.detach().clone()
    orig_x = x.detach().clone()

    # repeat with manual grad
    model.zero_grad()
    assert model.A.grad is None
    # re-enage gradients by computing x+f(x)
    x = model.inverse_with_manual_grad(y)
    x.norm().backward()
    new_grad = model.A.grad
    new_x = x.detach().clone()

    print(
        f"x error: {(orig_grad - new_grad).norm().item():.4f}"
        f" grad error: {(orig_grad - new_grad).norm().item():.4f}"
    )


model.debug = False
for k in range(10):
    # print(f"Test iteration {k}")
    test_grad()